# group by
그룹화와 집계 기능을 익힌다.
카테고리 평균, 지역별 매출 합계, 부서별 인원 수처럼 데이터를 묶어서 요약하는 작업을 수행한다.

## group by 기본
'groupby'는 같은 값을 가진 행끼리 묶는 기능이다.
묶은 뒤 평균, 합계, 개수 샅은 집계를 자주 수행한다.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# 직원 데이터 생성
employees = pd.DataFrame({
    '이름': ['김철수', '이영희', '박민수', '최지영', '정다은', '홍길동'],
    '부서': ['개발', '영업', '개발', '인사', '영업', '개발'],
    '급여': [5000, 4500, 5500, 4000, 4800, 5200]
})

employees  # 원본 데이터 확인

,이름,부서,급여
0,김철수,개발,5000
1,이영희,영업,4500
2,박민수,개발,5500
3,최지영,인사,4000
4,정다은,영업,4800
5,홍길동,개발,5200


In [3]:
# 부서별 평균 급여 계산
employees.groupby('부서')['급여'].mean()

부서
개발    5233.333333
영업    4650.000000
인사    4000.000000
Name: 급여, dtype: float64

## 여러 열 기준 그룹화
기준을 하나만 사용하는 것이 아니라 여러 컬럼을 함께 사용해 더 세분화 할 수 있다.

In [4]:
# 지역, 분기별 매출 데이터 생성
sales = pd.DataFrame({
    '지역': ['서울', '서울', '부산', '부산', '서울', '부산'],
    '분기': ['Q1', 'Q2', 'Q1', 'Q2', 'Q1', 'Q2'],
    '매출': [100, 120, 80, 90, 110, 85]
})

sales

,지역,분기,매출
0,서울,Q1,100
1,서울,Q2,120
2,부산,Q1,80
3,부산,Q2,90
4,서울,Q1,110
5,부산,Q2,85


In [ ]:
# 지역 분기별 매출 합계 (두 개의 기준으로 동시에 그룹화 한다)
# 지역 + 분기를 결합한 멀티 인덱스
sales.groupby(['지역','분기'])['매출'].sum()

지역  분기
부산  Q1     80
    Q2    175
서울  Q1    210
    Q2    120
Name: 매출, dtype: int64

In [7]:
# reset_index() : 그룹 기주이 인덱스로 올라간 상태를 다시 일반 열로 되돌린다.
sales.groupby(['지역','분기'])['매출'].sum().reset_index()

,지역,분기,매출
0,부산,Q1,80
1,부산,Q2,175
2,서울,Q1,210
3,서울,Q2,120


## 여러 집계 함수 사용
하나의 그룹에 대해 평균만 보는 것이 아니라 합계, 개수, 최대값, 최소값 등을 함께 구할 수 있다.

In [8]:
# 하나의 열에 여러 집계 함수 적용
employees.groupby('부서')['급여'].agg(['mean','sum','count'])

,mean,sum,count
부서,,,
개발,5233.333333,15700,3
영업,4650.000000,9300,2
인사,4000.000000,4000,1


In [9]:
# 열마다 다른 집계 함수 적용
# 급여는 평균/최대/최, 이름은 개수를 계산한다.
employees.groupby('부서').agg({
    '급여' : ['mean','max','min'],
    '이름' : 'count'
    })

급여                이름
           mean   max   min count
부서                               
개발  5233.333333  5500  5000     3
영업  4650.000000  4800  4500     2
인사  4000.000000  4000  4000     1

## named aggregation
집계 결과 컬럼명을 직접 지정하면 결과를 더 읽기 쉽게 만들 수 있다.

In [10]:
# 집계 결과 컬럼명 직접 지정
employees.groupby('부서').agg(
    평균급여=('급여','mean'),
    최고급여=('급여','max'),
    직원수=('이름','count') 
    )

,평균급여,최고급여,직원수
부서,,,
개발,5233.333333,5500,3
영업,4650.000000,4800,2
인사,4000.000000,4000,1


## filter()
'filter()'는 행 하나 하나를 거르는 것이 아니라, 그룹 전체를 남길지 말지를 결정하는 기능이다.

In [11]:
# 평균 급여가 5000 이상인 부서만 남기기 (조건을 만족하는 부서 그룹 전체가 유지 된다.)
employees.groupby('부서').filter(lambda x : x['급여'].mean() >= 5000)

,이름,부서,급여
0,김철수,개발,5000
2,박민수,개발,5500
5,홍길동,개발,5200


## transform()
'transform()'은 그룹별 계산 결과를 원본 행 개수에 맞춰 다시 돌려준다.
그래서 그룹 평균과 각 행을 비교하는 작업에 자주 사용된다.

In [13]:
# 부서 평균 급여를 원본 행 마다 붙이기
employees['부서평균'] = employees.groupby('부서')['급여'].transform('mean')

# 각 직원 급여가 부서 평균보다 얼마나 높은지 낮은지 계산
employees['평균대비'] = employees['급여'] - employees['부서평균']

employees

,이름,부서,급여,부서평균,평균대비
0,김철수,개발,5000,5233.333333,-233.333333
1,이영희,영업,4500,4650.000000,-150.000000
2,박민수,개발,5500,5233.333333,266.666667
3,최지영,인사,4000,4000.000000,0.000000
4,정다은,영업,4800,4650.000000,150.000000
5,홍길동,개발,5200,5233.333333,-33.333333


In [15]:
# 부서 내 급여 순위 계산
employees['부서내순위'] = employees.groupby('부서')['급여'].transform(
    'rank', ascending = False
)
employees

,이름,부서,급여,부서평균,평균대비,부서내순위
0,김철수,개발,5000,5233.333333,-233.333333,3.0
1,이영희,영업,4500,4650.000000,-150.000000,2.0
2,박민수,개발,5500,5233.333333,266.666667,1.0
3,최지영,인사,4000,4000.000000,0.000000,1.0
4,정다은,영업,4800,4650.000000,150.000000,1.0
5,홍길동,개발,5200,5233.333333,-33.333333,2.0


## apply
'apply()'는 그룹별로 조금 더 복잡한 로직을 직접 적용할 때 사용한다.
집계 하수 만으로 처리하기 어려울 때 유용하다.

In [16]:
# 각 부서에서 급여가 높은 상위 2명 추출 함수
def top_n(group, n=2):
    # nlargest: 특정 컬럼 기준으로 값이 큰 상위 n개 행 추출
    return group.nlargest(n,'급여')

employees.groupby('부서').apply(top_n,n=2)

이름    급여         부서평균        평균대비  부서내순위
부서                                             
개발 2  박민수  5500  5233.333333  266.666667    1.0
   5  홍길동  5200  5233.333333  -33.333333    2.0
영업 4  정다은  4800  4650.000000  150.000000    1.0
   1  이영희  4500  4650.000000 -150.000000    2.0
인사 3  최지영  4000  4000.000000    0.000000    1.0

## unstack/stack
groupby의 결과는 보통 계층형 인덱스를 가지는데, 'unstack()'과 'stack'으로 표의 모양으로 바꿀 수 있다.
- unstack : 인덱스를 열로 펼치기
- stack : 열을 다시 인덱스로 만들기

In [17]:
# 지역, 분기별 매출 합계 결과 저장
grouped = sales.groupby(['지역','분기'])['매출'].sum()
grouped

지역  분기
부산  Q1     80
    Q2    175
서울  Q1    210
    Q2    120
Name: 매출, dtype: int64

In [ ]:
# unstack() : 마지막 인덱스를 열로 펼친다.
wide = grouped.unstack()
wide

분기,Q1,Q2
지역,,
부산,80,175
서울,210,120


In [19]:
# stack() : 다시 세로로 쌓아서 원래 형태로 가깝게 되돌린다.
wide.stack()

지역  분기
부산  Q1     80
    Q2    175
서울  Q1    210
    Q2    120
dtype: int64